# Rankings Integration: World Football Elo + FIFA World Rankings

This notebook integrates two external team-strength signals into the processed ML dataset:

| Source | Coverage | Key Feature |
|--------|----------|-------------|
| **World Football Elo** (eloratings.net) | 1901–2023 | `home/away_elo_rating` |
| **FIFA World Rankings** (Kaggle / fifa.com) | Dec 1992–Jun 2024 | `home/away_fifa_rank`, `home/away_fifa_points` |

**Leakage prevention rule:** For each World Cup year, we use the rating/ranking snapshot from *before* the tournament started — never the end-of-year value.

**Year coverage note:** FIFA rankings begin in 1992, so matches before 1994 will have `NaN` for FIFA features. Elo covers all years. The team can filter to any year range after this join.

In [ ]:
import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR       = "../data"
ELO_PATH       = f"{DATA_DIR}/elo_ratings.csv"
FIFA_PATH      = f"{DATA_DIR}/fifa_ranking-2024-06-20.csv"
TEAMS_PATH     = f"{DATA_DIR}/teams.csv"           # official team ID → name mapping
MATCHES_PATH   = f"{DATA_DIR}/match_features_base.csv"
PROCESSED_PATH = f"{DATA_DIR}/processed/class_a_match_level.csv"
OUTPUT_PATH    = f"{DATA_DIR}/processed/class_a_with_rankings.csv"


## 1. Team ID → Team Name Mapping

The processed ML dataset uses team IDs (e.g. `T-31`). We need full names to join against Elo and FIFA data.

We use the **official `teams.csv`** from the [jfjelstul/worldcup](https://github.com/jfjelstul/worldcup) R package — the same source that generated the processed dataset. This covers all 88 team IDs across all World Cups from 1930–2022.

In [ ]:
# ── Load official team mapping from jfjelstul/worldcup ─────────────────────
teams_df = pd.read_csv(TEAMS_PATH)
team_id_to_name = teams_df.set_index("team_id")["team_name"].to_dict()

# Verify all team IDs in processed dataset are covered
processed_check = pd.read_csv(PROCESSED_PATH, usecols=["home_team_id", "away_team_id"])
all_ids = set(processed_check["home_team_id"]) | set(processed_check["away_team_id"])
unmapped = all_ids - set(team_id_to_name.keys())

print(f"Total teams in official CSV:         {len(team_id_to_name)}")
print(f"Unique team IDs in processed data:   {len(all_ids)}")
print(f"Unmapped IDs:                        {len(unmapped)}")
if unmapped:
    print(f"WARNING — missing: {sorted(unmapped)}")
else:
    print("All team IDs successfully resolved from official source.")

# Sample of pre-2006 teams now resolved correctly
sample_ids = ["T-05", "T-07", "T-10", "T-21", "T-60", "T-61", "T-72", "T-87"]
print("\nSample pre-2006 mappings from official CSV:")
for tid in sample_ids:
    print(f"  {tid} → {team_id_to_name.get(tid, 'NOT FOUND')}")


## 2. Load & Prepare World Football Elo Ratings

Source: eloratings.net  
Strategy: For a match in World Cup year `Y`, use the Elo rating from year `Y-1` (end-of-year snapshot before the tournament).

In [ ]:
elo = pd.read_csv(ELO_PATH)
elo["year"] = elo["year"].astype(int)

# Keep only the columns we need
elo = elo[["year", "team", "rating", "rank"]].copy()
elo.columns = ["elo_year", "team_name", "elo_rating", "elo_rank"]

# The join key: elo_year = tournament_year - 1 (pre-tournament snapshot)
# We create a lookup key = elo_year + 1 so we can merge on tournament year
elo["year"] = elo["elo_year"] + 1

# Handle duplicate ratings (keep highest rated entry per team per year)
elo = elo.sort_values("elo_rating", ascending=False).drop_duplicates(
    subset=["year", "team_name"]
)

print(f"Elo records: {len(elo):,}")
print(f"Tournament years covered (via shift): {elo['year'].min()} - {elo['year'].max()}")
print(elo.head())

### 2a. Team Name Normalization for Elo

Map project team names → Elo dataset team names to handle discrepancies.

In [ ]:
# project team name  →  Elo dataset team name
# Only needed where names differ. Elo already uses 'South Korea', 'Ivory Coast', 'Iran'.
ELO_NAME_MAP = {
    "United States":          "United States",   # same, explicit for clarity
    "North Korea":            "North Korea",
    "Dutch East Indies":      "Indonesia",
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
}

def normalize_for_elo(name):
    return ELO_NAME_MAP.get(name, name)

# Verify key teams exist in Elo data
sample_teams = ["France", "Brazil", "Germany", "South Korea", "Ivory Coast", "Iran"]
print("Elo coverage check:")
for t in sample_teams:
    found = t in elo["team_name"].values
    print(f"  {t}: {'FOUND' if found else 'MISSING'}")


## 3. Load & Prepare FIFA World Rankings

Source: Kaggle (cashncarry/fifaworldranking), updated June 2024  
Strategy: For each World Cup year, use the **last ranking snapshot published before the tournament kick-off**.

In [ ]:
fifa = pd.read_csv(FIFA_PATH)
fifa["rank_date"] = pd.to_datetime(fifa["rank_date"])
print(f"FIFA ranking records: {len(fifa):,}")
print(f"Date range: {fifa['rank_date'].min().date()} to {fifa['rank_date'].max().date()}")
print(f"Unique countries: {fifa['country_full'].nunique()}")
print(fifa.head())

In [ ]:
# World Cup start dates — we pick the last FIFA snapshot BEFORE each tournament
WC_START_DATES = {
    1994: "1994-06-17",
    1998: "1998-06-10",
    2002: "2002-05-31",
    2006: "2006-06-09",
    2010: "2010-06-11",
    2014: "2014-06-12",
    2018: "2018-06-14",
    2022: "2022-11-20",
}

fifa_snapshots = []
for wc_year, start_date in WC_START_DATES.items():
    cutoff = pd.Timestamp(start_date)
    pre_wc = fifa[fifa["rank_date"] < cutoff]
    if pre_wc.empty:
        print(f"WARNING: No FIFA snapshot found before {wc_year} WC")
        continue
    snapshot_date = pre_wc["rank_date"].max()
    snapshot = pre_wc[pre_wc["rank_date"] == snapshot_date].copy()
    snapshot["wc_year"] = wc_year
    fifa_snapshots.append(snapshot)
    print(f"{wc_year} WC → using FIFA snapshot from {snapshot_date.date()} ({len(snapshot)} teams)")

fifa_pre_wc = pd.concat(fifa_snapshots, ignore_index=True)
fifa_pre_wc = fifa_pre_wc[["wc_year", "country_full", "rank", "total_points"]].copy()
fifa_pre_wc.columns = ["year", "team_name", "fifa_rank", "fifa_points"]
print(f"\nTotal FIFA pre-WC records: {len(fifa_pre_wc):,}")

### 3a. Team Name Normalization for FIFA

Map project team names → FIFA dataset `country_full` names.

In [ ]:
# project team name  →  FIFA country_full name
FIFA_NAME_MAP = {
    "United States":          "USA",
    "South Korea":            "Korea Republic",
    "North Korea":            "Korea DPR",
    "Ivory Coast":            "Côte d'Ivoire",
    "Iran":                   "IR Iran",
    "Serbia and Montenegro":  "Serbia and Montenegro",
    "Czech Republic":         "Czech Republic",
    "Republic of Ireland":    "Ireland",
    "China":                  "China PR",
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Serbia":                 "Serbia",
}

def normalize_for_fifa(name):
    return FIFA_NAME_MAP.get(name, name)

# Verify key teams exist in FIFA data
sample_checks = ["USA", "Korea Republic", "Côte d'Ivoire", "France", "Brazil", "IR Iran"]
print("FIFA coverage check:")
for t in sample_checks:
    found = t in fifa_pre_wc["team_name"].values
    print(f"  {t}: {'FOUND' if found else 'MISSING'}")


## 4. Load Processed ML Dataset & Add Team Names

In [ ]:
processed = pd.read_csv(PROCESSED_PATH)
print(f"Processed dataset shape: {processed.shape}")
print(f"Year range: {processed['year'].min()} - {processed['year'].max()}")

# Map team IDs to names
processed["home_team_name"] = processed["home_team_id"].map(team_id_to_name)
processed["away_team_name"] = processed["away_team_id"].map(team_id_to_name)

# Report any still-unmapped teams
unmapped_home = processed[processed["home_team_name"].isna()]["home_team_id"].unique()
unmapped_away = processed[processed["away_team_name"].isna()]["away_team_id"].unique()
all_unmapped = set(unmapped_home) | set(unmapped_away)
if all_unmapped:
    print(f"WARNING: {len(all_unmapped)} team IDs still unmapped: {sorted(all_unmapped)}")
else:
    print("All team IDs successfully mapped to names.")

## 5. Join Elo Ratings

In [ ]:
# Normalize team names for Elo join
processed["home_name_elo"] = processed["home_team_name"].apply(normalize_for_elo)
processed["away_name_elo"] = processed["away_team_name"].apply(normalize_for_elo)

# Join home team Elo
elo_lookup = elo[["year", "team_name", "elo_rating", "elo_rank"]]

processed = processed.merge(
    elo_lookup.rename(columns={"team_name": "home_name_elo",
                                "elo_rating": "home_elo_rating",
                                "elo_rank":   "home_elo_rank"}),
    on=["year", "home_name_elo"],
    how="left"
)

# Join away team Elo
processed = processed.merge(
    elo_lookup.rename(columns={"team_name": "away_name_elo",
                                "elo_rating": "away_elo_rating",
                                "elo_rank":   "away_elo_rank"}),
    on=["year", "away_name_elo"],
    how="left"
)

# Derived feature: difference in Elo (home perspective)
processed["feat_elo_rating_diff"] = processed["home_elo_rating"] - processed["away_elo_rating"]

elo_coverage = processed["home_elo_rating"].notna().mean()
print(f"Elo coverage: {elo_coverage:.1%} of matches have home Elo rating")
print(processed[["year", "home_elo_rating", "away_elo_rating", "feat_elo_rating_diff"]].head(10))

## 6. Join FIFA Rankings

In [ ]:
# Normalize team names for FIFA join
processed["home_name_fifa"] = processed["home_team_name"].apply(normalize_for_fifa)
processed["away_name_fifa"] = processed["away_team_name"].apply(normalize_for_fifa)

fifa_lookup = fifa_pre_wc[["year", "team_name", "fifa_rank", "fifa_points"]]

# Join home team FIFA
processed = processed.merge(
    fifa_lookup.rename(columns={"team_name":   "home_name_fifa",
                                 "fifa_rank":   "home_fifa_rank",
                                 "fifa_points": "home_fifa_points"}),
    on=["year", "home_name_fifa"],
    how="left"
)

# Join away team FIFA
processed = processed.merge(
    fifa_lookup.rename(columns={"team_name":   "away_name_fifa",
                                 "fifa_rank":   "away_fifa_rank",
                                 "fifa_points": "away_fifa_points"}),
    on=["year", "away_name_fifa"],
    how="left"
)

# Derived features: rank difference (lower = better, so away - home = positive means home is better ranked)
processed["feat_fifa_rank_diff"]   = processed["away_fifa_rank"]   - processed["home_fifa_rank"]
processed["feat_fifa_points_diff"] = processed["home_fifa_points"] - processed["away_fifa_points"]

fifa_coverage = processed["home_fifa_rank"].notna().mean()
print(f"FIFA coverage: {fifa_coverage:.1%} of matches have home FIFA rank")
print("\nFIFA NaN by year (expected NaN for pre-1994 years):")
print(processed.groupby("year")["home_fifa_rank"].apply(lambda x: x.isna().sum()).rename("missing_fifa").to_string())

## 7. Coverage Summary

In [ ]:
new_features = [
    "home_elo_rating", "away_elo_rating", "home_elo_rank", "away_elo_rank", "feat_elo_rating_diff",
    "home_fifa_rank", "away_fifa_rank", "home_fifa_points", "away_fifa_points",
    "feat_fifa_rank_diff", "feat_fifa_points_diff",
]

print("New feature coverage (non-null %):\n")
coverage = processed[new_features].notna().mean().mul(100).round(1)
print(coverage.to_string())

print(f"\nFull dataset shape: {processed.shape}")
print(f"Original columns: 104  →  New columns: {processed.shape[1]}")
print(f"New features added: {processed.shape[1] - 104}")

## 8. Drop Intermediate Name Columns & Save

In [ ]:
# Drop the temporary name columns used for joining
cols_to_drop = ["home_team_name", "away_team_name", "home_name_elo", "away_name_elo",
                "home_name_fifa", "away_name_fifa"]
processed = processed.drop(columns=cols_to_drop)

# Save
processed.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")
print(f"Final shape: {processed.shape}")

# Quick sanity check on 2022 WC
print("\nSanity check — 2022 WC sample:")
cols_check = ["year", "home_team_id", "away_team_id",
              "home_elo_rating", "away_elo_rating", "feat_elo_rating_diff",
              "home_fifa_rank", "away_fifa_rank", "feat_fifa_rank_diff"]
print(processed[processed["year"] == 2022][cols_check].head(8).to_string())

## 9. Year Range Filtering (Placeholder)

The team will decide on the final year range. Uncomment and set the range below to filter the dataset.

In [ ]:
# ── Uncomment after year range is agreed upon ──────────────────────────────
# YEAR_START = 1994
# YEAR_END   = 2022
#
# filtered = processed[(processed["year"] >= YEAR_START) & (processed["year"] <= YEAR_END)]
# print(f"Filtered dataset: {filtered.shape[0]} matches ({YEAR_START}-{YEAR_END})")
# filtered.to_csv(f"{DATA_DIR}/processed/class_a_with_rankings_{YEAR_START}_{YEAR_END}.csv", index=False)

print("Year range filtering not applied yet — full dataset saved.")
print(f"\nMatches by World Cup year:")
print(processed.groupby("year").size().rename("n_matches").to_string())